In [ ]:
import openai
from opensearchpy import OpenSearch
import torch
from sentence_transformers import SentenceTransformer 
import json
import time
from tqdm import tqdm
import pandas as pd
import os
import random
from collections import Counter

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model = SentenceTransformer("intfloat/multilingual-e5-base", device=device) # MODELO DAS EMBEDDINGS

In [ ]:
client = OpenSearch(
    hosts=[{'host': 'seuhost.com.br', 'port': 8000}],
    http_auth=('xxxxxx', 'xxxxxx'),
    use_ssl=True,
    verify_certs=False, 
    timeout=30)

In [ ]:
def run_query(query):
    query = 'query: ' + query

    emb_qry = model.encode(query, show_progress_bar=False)
    emb_qry = emb_qry.tolist()

    query_body = {"size": 20,
        "query": {"knn": {"embedding": {"vector": emb_qry, "k": 10}}},
        "_source": False,
            "fields": ["docid", "json_name", "text"],
    }
    
    response = client.search(
        body = query_body,
        index = 'regis1'
    )
    chunks = []
    for j, hit in enumerate(response["hits"]["hits"]):
        chunks.append({
            'index': hit['_index'],
            'order': j,
            'id': hit['_id'],
            'score': hit['_score'],
            'chunk': hit['fields']['text'][0]
        })
    return chunks

In [ ]:
def generate_llm_response(prompt, model):

    client = openai.OpenAI(
        api_key= "sk-XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
    )
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role":"user","content": prompt},
        ]
    )
    return completion.choices[0].message.content

def make_prompt(prompt_file, completion, model='gpt-4o-mini'):
    with open(prompt_file) as pf:
        pre_prompt = pf.read()
        return generate_llm_response(pre_prompt+completion,model)

In [ ]:
def try_to_decode_json(str):
    try:
        ans = json.loads(str)
    except:
        ans = json.loads(str[8:-4])
    return ans

In [ ]:
seeds = [
    "Como a dolomita afeta a porosidade e permeabilidade de um reservatório carbonático?",
    "Quais são os efeitos da halocinese na migração secundária de hidrocarbonetos?",
    "Como a sísmica 3D pode ser usada para mapear domos e diapiros salinos?",
    "Quais técnicas de perfilagem de poços são mais eficazes na caracterização dos reservatórios do Membro Mucuri?",
    "Como a variação morfológica dos ostracodes pode indicar mudanças paleoambientais em bacias sedimentares?"
]

selected_chunks = []
for seed in seeds:
    selected_chunks.append(run_query(seed)[0])

In [ ]:
def make_factual_questions_for_chunks(chunks):
    
    ans = []
    for chunk in tqdm(chunks):
        time.sleep(3)
        response = make_prompt('prompts/factual.txt',
                               f"\nTexto de Entrada:{{{chunk['chunk']}}}")
        response = try_to_decode_json(response)
        for i, qa_pair in enumerate(response['qa_pairs']):
            ans.append({
                'chunk_id':chunk['id'],
                'chunk': chunk['chunk'],
                'question_num': i,
                'question':qa_pair['question'],
                'answer':qa_pair['answer']
            })
    return ans

xx = make_factual_questions_for_chunks(selected_chunks)

In [ ]:
pd.DataFrame.from_records(xx).to_csv('25_factuais.csv')